# Lab F: LangGraph Agent (NovaPay Fraud Investigation)

**Advanced FDE Lab** | The same NovaPay fraud agent built as a LangGraph state machine.

LangGraph models agents as **directed graphs**: nodes do work, edges decide the path, and conditional edges create the reason-act-observe loop. You get checkpointing, streaming, and human-in-the-loop pause/resume for free.

**What you'll learn:**

- Build a production agent as a LangGraph StateGraph
- Nodes, edges, conditional routing (the graph-based agent loop)
- Checkpointing (persist state, resume after crash)
- Human-in-the-loop interrupt (pause before dangerous actions)
- Compare: LangGraph vs Strands (when to pick which)

**NovaPay story:** Same fraud investigation as the other labs, but as a graph.

In [ ]:
%pip install -q -r requirements.txt
import json, time, uuid
from typing import TypedDict, Annotated, Literal
from IPython.display import HTML, display

def section(t):
    display(HTML(f'<div style="border-left:5px solid #4b2e83;padding:8px 14px;margin-top:16px;font-family:Arial;font-size:1.25em;font-weight:700">{t}</div>'))

def tip(t):
    display(HTML(f'<div style="background:#eef7ff;border:1px solid #1f6feb;border-radius:6px;padding:10px 16px;margin:8px 0;font-family:Arial"><b>Tip:</b> {t}</div>'))

print("Ready.")

In [ ]:
section("1. The NovaPay tools (same business logic, framework-independent)")

# These are the real NovaPay tools - same data regardless of framework
TRANSACTIONS = {
    "TXN-101": {"amount": 420.0, "merchant": "ElectroWorld", "country": "GB", "customer": "CUST-1001"},
    "TXN-102": {"amount": 89.99, "merchant": "Amazon", "country": "US", "customer": "CUST-1001"},
    "TXN-103": {"amount": 1450.0, "merchant": "GiftCardHub", "country": "GB", "customer": "CUST-1001"},
}
CUSTOMERS = {"CUST-1001": {"name": "Amara Okafor", "home_country": "US", "tier": "premier"}}

def tool_get_transaction(txn_id: str) -> str:
    txn = TRANSACTIONS.get(txn_id)
    return json.dumps(txn) if txn else json.dumps({"error": "not found"})

def tool_check_fraud(txn_id: str) -> str:
    txn = TRANSACTIONS.get(txn_id)
    if not txn: return json.dumps({"error": "not found"})
    cust = CUSTOMERS.get(txn["customer"], {})
    score = 0
    signals = []
    if txn["amount"] > 500: score += 40; signals.append("high_amount")
    if txn["country"] != cust.get("home_country", "US"): score += 30; signals.append("geo_mismatch")
    if txn["merchant"] in ["GiftCardHub", "ElectroWorld"]: score += 30; signals.append("risky_merchant")
    return json.dumps({"txn_id": txn_id, "risk_score": min(score, 100), "signals": signals,
                        "recommendation": "freeze" if score >= 70 else "monitor" if score >= 40 else "allow"})

def tool_freeze_card(customer_id: str, reason: str) -> str:
    return json.dumps({"customer_id": customer_id, "action": "FROZEN", "reason": reason})

print("NovaPay tools defined (framework-independent)")

In [ ]:
section("2. Build the LangGraph agent")
tip("A LangGraph agent is a StateGraph: typed state flows through nodes connected by edges. A conditional edge after the model node creates the ReAct loop.")

from langgraph.graph import StateGraph, END
from langchain_aws import ChatBedrock
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool

# Wrap our tools for LangChain
@tool
def get_transaction(txn_id: str) -> str:
    """Look up a NovaPay transaction by ID."""
    return tool_get_transaction(txn_id)

@tool
def check_fraud(txn_id: str) -> str:
    """Check fraud signals for a transaction. Returns risk score and signals."""
    return tool_check_fraud(txn_id)

@tool
def freeze_card(customer_id: str, reason: str) -> str:
    """Freeze a customer's card. HIGH-RISK ACTION - requires justification."""
    return tool_freeze_card(customer_id, reason)

TOOLS = [get_transaction, check_fraud, freeze_card]
TOOL_MAP = {t.name: t for t in TOOLS}

# The LLM (Bedrock Haiku via LangChain)
llm = ChatBedrock(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0", region_name="us-east-1")
llm_with_tools = llm.bind_tools(TOOLS)

print("LLM + tools bound. Now building the graph...")

In [ ]:
# Define the graph state
from typing import Sequence
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

# Node: call the model
def call_model(state: AgentState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# Node: execute tool calls
def call_tools(state: AgentState) -> dict:
    last_msg = state["messages"][-1]
    results = []
    for tc in last_msg.tool_calls:
        tool_fn = TOOL_MAP[tc["name"]]
        result = tool_fn.invoke(tc["args"])
        results.append(ToolMessage(content=result, tool_call_id=tc["id"]))
    return {"messages": results}

# Conditional edge: should we call tools or finish?
def should_continue(state: AgentState) -> Literal["tools", "end"]:
    last_msg = state["messages"][-1]
    if last_msg.tool_calls:
        return "tools"
    return "end"

# Build the graph
graph = StateGraph(AgentState)
graph.add_node("model", call_model)
graph.add_node("tools", call_tools)
graph.set_entry_point("model")
graph.add_conditional_edges("model", should_continue, {"tools": "tools", "end": END})
graph.add_edge("tools", "model")  # loop back after tool execution

# Compile
app = graph.compile()
print("LangGraph agent compiled!")
print("  Nodes: model, tools")
print("  Edges: model -> (tools | END), tools -> model")
print("  This IS the ReAct loop, expressed as a graph.")

In [ ]:
section("3. Run the agent")

result = app.invoke({"messages": [HumanMessage(content="Check transaction TXN-103 for fraud. If risk is high, freeze the card for CUST-1001.")]})

print("Agent execution complete. Messages:")
print("=" * 60)
for msg in result["messages"]:
    role = msg.__class__.__name__
    content = msg.content[:200] if isinstance(msg.content, str) else str(msg.content)[:200]
    print(f"  [{role}] {content}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"    -> tool: {tc['name']}({tc['args']})")

In [ ]:
section("4. Checkpointing (persist + resume)")
tip("LangGraph can checkpoint state after every node. If the process crashes mid-execution, it resumes from the last checkpoint — critical for long-running agents.")

from langgraph.checkpoint.memory import MemorySaver

# Compile with checkpointing
checkpointer = MemorySaver()
app_with_checkpoints = graph.compile(checkpointer=checkpointer)

# Run with a thread_id (the session key)
config = {"configurable": {"thread_id": "novapay-session-001"}}
result = app_with_checkpoints.invoke(
    {"messages": [HumanMessage(content="What are the fraud signals on TXN-101?")]},
    config=config
)
print("Run 1 complete. State checkpointed.")
print(f"  Last message: {result['messages'][-1].content[:100]}")

# Continue the SAME session (it remembers the context)
result2 = app_with_checkpoints.invoke(
    {"messages": [HumanMessage(content="Now check TXN-103 as well.")]},
    config=config
)
print(f"\nRun 2 (same session, remembers context):")
print(f"  Last message: {result2['messages'][-1].content[:100]}")
print(f"  Total messages in session: {len(result2['messages'])}")

In [ ]:
section("5. Human-in-the-loop interrupt")
tip("For dangerous actions (freeze_card), the graph pauses BEFORE executing and waits for human approval. This is built into LangGraph's interrupt mechanism.")

from langgraph.prebuilt import ToolNode

# Rebuild with interrupt_before on the tools node
def should_interrupt(state: AgentState) -> Literal["tools", "end"]:
    last_msg = state["messages"][-1]
    if last_msg.tool_calls:
        # Check if any tool call is high-risk
        for tc in last_msg.tool_calls:
            if tc["name"] == "freeze_card":
                print(f"  INTERRUPT: freeze_card requested. Awaiting human approval.")
        return "tools"
    return "end"

graph_hitl = StateGraph(AgentState)
graph_hitl.add_node("model", call_model)
graph_hitl.add_node("tools", call_tools)
graph_hitl.set_entry_point("model")
graph_hitl.add_conditional_edges("model", should_interrupt, {"tools": "tools", "end": END})
graph_hitl.add_edge("tools", "model")

# Compile with interrupt_before on tools (pauses before executing any tool)
hitl_checkpointer = MemorySaver()
app_hitl = graph_hitl.compile(checkpointer=hitl_checkpointer, interrupt_before=["tools"])

config2 = {"configurable": {"thread_id": "hitl-session-001"}}
print("Running with interrupt_before=['tools']...")
result = app_hitl.invoke(
    {"messages": [HumanMessage(content="TXN-103 is fraud. Freeze the card for CUST-1001.")]},
    config=config2
)
print(f"  Graph PAUSED. Model wants to call: {result['messages'][-1].tool_calls[0]['name'] if result['messages'][-1].tool_calls else 'none'}")
print(f"  In production: send to approval queue, wait for human, then resume with app_hitl.invoke(None, config2)")

# Resume (simulating human approved)
print("\n  [HUMAN APPROVED] Resuming...")
final = app_hitl.invoke(None, config2)
print(f"  Final: {final['messages'][-1].content[:150]}")

In [ ]:
section("6. LangGraph vs Strands: when to pick which")

print("""
LANGGRAPH vs STRANDS - FDE DECISION FRAMEWORK:

| Dimension | LangGraph | Strands |
|-----------|-----------|---------|
| Agent model | Explicit GRAPH (nodes + edges) | Implicit LOOP (@tool + Agent()) |
| Control over flow | Total (you define every edge) | High (system prompt + tools) |
| Checkpointing | Built-in (MemorySaver, Postgres) | FileSessionManager |
| Human-in-the-loop | interrupt_before/after (native) | Manual (flag + approval tool) |
| Streaming | Built-in per-node | Built-in |
| Multi-agent | Subgraphs + edges | Agent-as-a-tool |
| AWS integration | langchain-aws (ChatBedrock) | Native BedrockModel |
| Learning curve | Higher (graph concepts) | Lower (just Agent + @tool) |
| Best for | Complex flows with branching, HITL, checkpointing needs | Straightforward tool-using agents, quick prototypes |
| Production deployment | LangGraph Platform / self-hosted | AgentCore Runtime |

WHEN TO PICK LANGGRAPH:
- You need explicit control over the execution path
- Checkpointing / resume-after-crash is critical
- Human-in-the-loop is a core requirement (interrupt is native)
- The flow has complex branching (not just "call tools until done")

WHEN TO PICK STRANDS:
- You're on AWS and want native Bedrock + AgentCore integration
- The agent is straightforward (tools + system prompt)
- You want the simplest possible code (3 lines to a working agent)
- Multi-agent via agent-as-a-tool is sufficient
""")

In [ ]:
section("7. Knowledge check")
print("""
Q1. What is the 'conditional edge' in a LangGraph agent?
   -> The decision point after the model node: if tool_calls exist, route to 'tools';
      otherwise route to END. This IS the ReAct loop.

Q2. What does checkpointing give you that a plain loop doesn't?
   -> State persisted after every node. Process can crash and resume from last checkpoint.
      Critical for long-running agents (minutes/hours) and for HITL pause/resume.

Q3. How does LangGraph's interrupt_before differ from a manual HITL gate?
   -> It's built into the graph runtime: execution pauses at the specified node,
      state is checkpointed, and the graph resumes when you call invoke(None, config).
      No custom queue/polling code needed.

Q4. When would you pick LangGraph over Strands?
   -> When you need explicit flow control, native checkpointing, or built-in HITL interrupts.
      Strands is simpler for basic tool-using agents on AWS.
""")